# Lesson 3.8e — 最小多模态融合模型（Minimal Multimodal Policy）

3.8.1–3.8.4 一直在讨论**数据应该是什么**、**哪些 state 可以给**、**语言什么时候有意义**、
**如何证明模型真的使用语言**。现在终于实现第一次：

$$\boxed{\ \pi_\theta(I_t,\ p_t,\ g,\ \ell)\ \rightarrow\ a_{t:t+H-1}\ }$$

**目标不是复现 π₀.₅。** 目标是搭一个你**完全理解**的 mini-VLA：
输入多个模态 → 编码 → 融合 → 输出 action chunk。失败时能指出是哪一段坏掉。

### 结构：Encoder + Fusion + Action Head

```text
   Image [3,128,128]        Proprio [25]      Goal [3]        Language [8] ids
          |                      |                |                  |
    CNN Encoder            MLP Encoder      MLP Encoder      Embedding + mean pool
          |                      |                |                  |
     z_I [256]              z_P [128]         z_G [32]          z_L [32]
          \\______________________|________________|__________________/
                                     |
                              concat  z [448]
                                     |
                              Fusion MLP  448 -> 512 -> 256
                                     |
                            Action Head  256 -> 64   (H x 8)
                                     |
                              action_chunk [8, 8]
```

### 为什么第一版**不**用 cross-attention

不是因为"先用简单的练手"，而是因为**拼接式融合的能力边界我们在 3.8.4.5 已经量过了**：

| | 要求 | 本架构 |
|---|---|---|
| (a) 视觉保留 **token 网格** | $Z_I \in \mathbb{R}^{n_{\text{patch}}\times d}$ | ❌ 池化成一个向量 |
| (b) 有 attention 步让语言调制读哪些 patch | cross-attention / joint self-attention | ❌ 只有 concat + MLP |

两条都不满足，所以它**原理上做不了指称 grounding**（"red" 无法绑定到某块像素），
**只能做动作模式 grounding**（指令 → 哪种行为模式）。

而"动作模式 grounding"正好是**本数据唯一能测的那一种**（3.8.4.5：指令里没有指称词，
每场景只有一个物体）。所以第一版的能力边界与数据的可测范围**恰好对齐**——
这就是"从最简单可验证模型开始"在这里的精确含义，不是一句口号。

**它还带来一个方法学好处**：直接上 ViT + LLM tokenizer + cross-attention + action transformer，
失败时你分不清是 image encoder、language encoder、attention、action head 还是数据的问题。

## 运行说明

本 notebook 覆盖 **3.8.5 最小融合模型**：数据集、模型、前向/反向/训练，以及四个模态开关。

- 契约在 `scripts/mml_contract.py`；模型与训练循环在 `scripts/mml_policy.py`。两者都是**单一实现**，
  3.8.6 的消融会 import 同一份，所以不会出现"两套模型定义各自漂移"。
- 执行顺序：`Kernel → Restart Kernel and Run All Cells`。
- **CPU**：本机 `torch.cuda.is_available()` 为 `False`。数据集只有 637 个样本，规模是刻意压到 CPU 可训的。
- 3.8.6 才做三个模态臂的**完整对比**；本 notebook 只训练一个（全模态）模型来验证 pipeline 通。

**命名**：`pick` / `push` 是 episode 列表；任务级字典写作 `datasets["PickCube-v1"]`。

In [7]:
# 前置：契约来自 scripts/mml_contract.py，模型与训练循环来自 scripts/mml_policy.py —— 都是单一实现。
import logging
import sys
import warnings
from pathlib import Path

import numpy as np
import torch

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / ".git").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "scripts"))
import mml_contract as mmc
import mml_policy as mp

logging.getLogger("mani_skill").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", message=".*NVML.*")
warnings.filterwarnings("ignore", message=".*CUDA initialization.*")
warnings.filterwarnings("ignore", message=".*cudaGetDeviceCount.*")

datasets = mmc.load_datasets()
INSTRUCTIONS, VOCAB, T_TXT, PAD = mmc.INSTRUCTIONS, mmc.VOCAB, mmc.T_TXT, mmc.PAD
H, ACTION_DIM = mmc.H, mmc.ACTION_DIM
SEED = 0
torch.manual_seed(SEED)

print(f"torch {torch.__version__} | device {mp.DEVICE}")
print(f"contract: {len(datasets)} tasks, "
      f"{sum(len(d['episodes']) for d in datasets.values())} episodes, H = {H}")

torch 2.11.0+cu128 | device cuda
contract: 2 tasks, 10 episodes, H = 8


## 3.8.5.1 四个编码器的选择，以及被否决的两个

| 模态 | 本版选择 | 输出 | 理由 |
|---|---|---|---|
| **image** | **小型 CNN，从头训练** | 256 | 见下 |
| **proprio** | MLP 25→128→128 | 128 | 低维、已经是语义量，不需要 encoder |
| **goal** | MLP 3→32 | 32 | 同 proprio。**不直接 concat**——3 维与 256 维的尺度差两个数量级 |
| **language** | **learned token embedding + masked mean pool** | 32 | 见下 |

### 为什么 image 不用 pretrained ResNet18

三条理由，第一条是硬的：

1. **本机没有缓存，pretrained 需要联网下载。** `~/.cache/torch/hub/checkpoints/` 不存在，
   而这次会话里网络一直在失败（`git push` 重试了 5 次）。一个需要下载权重的第一版模型
   **不可复现**。
2. **你自己的论证对它同样成立。** 你反对 Option C（预训练 text encoder）的理由是"引入变量：
   tokenizer / pretrained distribution / frozen-finetune"。**pretrained image encoder 引入的是
   同一类变量**（ImageNet 分布、归一化约定、BN 的 train/eval 行为）。在**验证 pipeline** 的
   模型里引入它，与你的原则不一致。
3. **规模不匹配。** ResNet18 有 11M 参数，而训练集只有 **509** 个样本。而且它的
   residual / BatchNorm / bottleneck 约定你无法逐层解释——这与你"建立一个完全理解的 mini-VLA"
   的目标直接冲突。四层 stride-2 卷积 + global average pooling 你能讲清每一层在做什么。

**为什么必须用 CNN 而不是 flatten**：`3×128×128 = 49152`，接 `Linear(49152, 256)` 单层就是
**12.6M 权重**（比整个 ResNet18 还大），而且丢掉了空间的归纳偏置。CNN 天然学 edge → shape → object。

### 为什么 language 不用 one-hot task id

**决定性理由：one-hot 会让 3.8.4.6 的三级阶梯坍缩。** T2 测的是**词序**置换——
one-hot 向量**没有词序可打乱**，T2 直接失去定义，三级阶梯退回两级。

而且这个替换**不损失任何能力**：只有 2 个任务时 L0（任务编号）与 L1（token embedding）
**信息等价**（3.8.4.4：存在双射），所以 L0 是 L1 的**上界**而非更强的表示。
换成 token embedding 是**零信息代价**的，同时保住了三级阶梯，并且这条代码路径正是
3.8.7 接 pretrained text encoder 时要保留的那条。

## 3.8.5.2 数据集：chunk 样本严格在 episode 内

沿用 3.7 的构造方式：**丢掉每个 episode 末尾的 $H-1$ 步**，而不是补零。

$$\text{样本数} = \sum_i T_i - N(H-1) = 637$$

补零会往训练集里塞进"未来是静止的"这种在数据里不存在的样本，而且会跨 episode 边界。
两条断言把这个性质钉住：每个 chunk 必须**恰好等于**它自己 episode 的那一段切片，且不得越界。

In [8]:
# 3.8.5.2 数据集
raw = mp.build_dataset(datasets, H=H)

print(f"{'field':<14} {'shape':<22} dtype")
print("-" * 52)
for k in mp.FIELDS:
    print(f"{k:<14} {str(raw[k].shape):<22} {raw[k].dtype}")

print()
for env_id in sorted(datasets):
    n = int((raw["task_of"] == env_id).sum())
    per_ep = [int(((raw["task_of"] == env_id) & (raw["episode_of"] == i)).sum())
              for i in range(len(datasets[env_id]["episodes"]))]
    print(f"  {env_id:>13}: {n:>4} samples | per episode {per_ep}")

total = sum(len(e["action"]) for d in datasets.values() for e in d["episodes"])
n_ep = sum(len(d["episodes"]) for d in datasets.values())
print()
print(f"  sum T = {total}, N = {n_ep}, N(H-1) = {n_ep * (H - 1)}")
assert len(raw["start_of"]) == total - n_ep * (H - 1), "the chunk count does not match the formula"
print(f"  -> {total} - {n_ep}*{H - 1} = {len(raw['start_of'])} samples, and no chunk crosses an episode")
print()
print("  (build_dataset asserts, per sample, that action_chunk == its own episode's slice and")
print("   that t + H <= T. Padding would have broken both.)")

field          shape                  dtype
----------------------------------------------------
image          (637, 128, 128, 3)     uint8
proprio        (637, 25)              float32
goal           (637, 3)               float32
language_ids   (637, 8)               int64
task_index     (637,)                 int64
action_chunk   (637, 8, 8)            float32

    PickCube-v1:  325 samples | per episode [67, 67, 43, 79, 69]
    PushCube-v1:  312 samples | per episode [64, 65, 57, 67, 59]

  sum T = 707, N = 10, N(H-1) = 70
  -> 707 - 10*7 = 637 samples, and no chunk crosses an episode

  (build_dataset asserts, per sample, that action_chunk == its own episode's slice and
   that t + H <= T. Padding would have broken both.)


## 3.8.5.3 划分与归一化

**划分是 episode 级的，不是 frame 级的。** 沿用 3.7 的约定：`default_rng(42).permutation(n)`，
**`order[0]` 是被留出的那一集**（3.7 记录过 `permutation(5)[0] == 4`）。这里对**每个任务各留一集**，
所以留出的仍然是前几课认定的那些 episode。

**归一化只在训练 episode 的原始帧上拟合**，绝不用 chunk 样本、绝不用验证 episode。

**而且是跨任务 pooled 的——这一条是承重的：**

3.8.4.5 量过，夹爪的均值动作 PickCube 是 `+0.5120`、PushCube 是 `−1.0000`，
任务相关方差有 **99.54%** 集中在这一个维度。**如果按任务分别归一化，两个任务的夹爪都会被映射到
零均值，那个信号就被抹掉了。** 下面这条断言就是守这个的。

In [9]:
# 3.8.5.3 划分与归一化
train_mask, val_mask, held = mp.episode_split(datasets, raw, seed=42)
print(f"train {train_mask.sum()} samples | val {val_mask.sum()} samples | held out {held}")
assert np.random.default_rng(42).permutation(5)[0] == 4, "3.7's held-out convention changed"
for env_id, val_ids in held.items():
    assert val_ids == [4], f"{env_id}: expected episode 4 held out, got {val_ids}"
print("  episode-level, no frame leakage, one episode per task held out")

norm = mp.fit_normalization(datasets, held)
print(f"\nnormalisation fitted on {norm['n_frames']} TRAINING frames only (pooled across tasks)")

# 承重断言：跨任务 pooled 归一化之后，夹爪的任务差异必须仍然存在
a_mean, a_std = (t.numpy() for t in norm["action"])
print(f"  pooled gripper action: mean {a_mean[7]:+.4f}  std {a_std[7]:.4f}")
norm_grip = {}
for env_id in sorted(datasets):
    g = np.concatenate([e["action"][:, 7] for e in datasets[env_id]["episodes"]])
    norm_grip[env_id] = (g.mean() - a_mean[7]) / a_std[7]
    print(f"  {env_id:>13}: raw mean {g.mean():+.4f} -> normalised {norm_grip[env_id]:+.4f}")
spread = abs(norm_grip["PickCube-v1"] - norm_grip["PushCube-v1"])
assert spread > 1.0, f"the task signal was erased by normalisation (spread {spread:.3f})"
print(f"  -> normalised gripper means differ by {spread:.3f} and have opposite signs")
print(f"     per-task normalisation would have made both ~0")

data = mp.apply_normalization(raw, norm)
assert data["image"].dtype == np.uint8, "images must stay uint8; the model scales them"
assert data["action_chunk"].dtype == np.float32
print(f"\nnormalised arrays ready: proprio/goal/action float32, image stays uint8 [0,255]")

train 509 samples | val 128 samples | held out {'PickCube-v1': [4], 'PushCube-v1': [4]}
  episode-level, no frame leakage, one episode per task held out

normalisation fitted on 565 TRAINING frames only (pooled across tasks)
  pooled gripper action: mean -0.4513  std 0.8924
    PickCube-v1: raw mean +0.0500 -> normalised +0.5618
    PushCube-v1: raw mean -1.0000 -> normalised -0.6149
  -> normalised gripper means differ by 1.177 and have opposite signs
     per-task normalisation would have made both ~0

normalised arrays ready: proprio/goal/action float32, image stays uint8 [0,255]


## 3.8.5.4 模型：Encoder + Concat Fusion + Action Head

| 段 | 层 | 输出 |
|---|---|---|
| Image encoder | `Conv(3→16,s2)` `Conv(16→32,s2)` `Conv(32→64,s2)` `Conv(64→128,s2)` `AdaptiveAvgPool2d(1)` `Linear(128→256)` | `[B,256]` |
| Proprio encoder | `Linear(25→128)` `Linear(128→128)` | `[B,128]` |
| Goal encoder | `Linear(3→32)` | `[B,32]` |
| Language encoder | `Embedding(10→16)` → masked mean pool → `Linear(16→32)` | `[B,32]` |
| Fusion | `concat` → `Linear(448→512)` `Linear(512→256)` | `[B,256]` |
| Action head | `Linear(256→64)` → `view(B,8,8)` | `[B,8,8]` |

**没有 BatchNorm**：少一个 train/eval 行为差异，第一版不值得引入。
**language 用 masked mean pool**：`padding_idx` 已经把 `<pad>` 的嵌入压成零向量，mask 是第二道保险。

下面这个 cell 做三件事：数参数、**把 `F.mse_loss` 的静默 broadcast 钉死**、以及打印一次逐层 shape trace。

In [10]:
# 3.8.5.4 模型
model = mp.MultimodalPolicy(vocab_size=len(VOCAB), t_txt=T_TXT)
print(f"trainable parameters: {mp.count_params(model):,}")
print(f"fusion input dims {model.fusion_dims} -> {sum(model.fusion_dims)}")
assert sum(model.fusion_dims) == 448, "the published concatenation dim changed"

loader = torch.utils.data.DataLoader(mp.DictDataset(data, train_mask), batch_size=32, shuffle=True)
batch = next(iter(loader))
pred = model(batch)
target = batch["action_chunk"]

# F.mse_loss 会静默 broadcast：[B,8,8] 对 [B,8] 会算出一个 [B,B,8] 的垃圾 loss 而不报错。
# 守卫是断言，不是注释。
assert pred.shape == target.shape == (len(target), H, ACTION_DIM), (pred.shape, target.shape)
print(f"\nforward: image{tuple(batch['image'].shape)} + proprio{tuple(batch['proprio'].shape)}"
      f" + goal{tuple(batch['goal'].shape)} + lang{tuple(batch['language_ids'].shape)}"
      f" -> {tuple(pred.shape)}")
assert pred.shape == target.shape, "F.mse_loss would now broadcast silently"

# 反向也要真的通
loss = torch.nn.functional.mse_loss(pred, target)
loss.backward()
grads = [p.grad for p in model.parameters() if p.grad is not None]
assert len(grads) == sum(1 for _ in model.parameters()), "some parameter received no gradient"
assert all(torch.isfinite(g).all() for g in grads), "a gradient is not finite"
model.zero_grad()
print(f"backward: {len(grads)} parameter tensors received finite gradients")

print("\n逐层 shape trace（信息路径，不是构造函数）：")
print(f"  {'module':<30} {'type':<18} shape")
for name, kind, shape in mp.trace_shapes(model, batch):
    print(f"  {name:<30} {kind:<18} {shape}")

trainable parameters: 528,800
fusion input dims [256, 128, 32, 32] -> 448

forward: image(32, 128, 128, 3) + proprio(32, 25) + goal(32, 3) + lang(32, 8) -> (32, 8, 8)
backward: 25 parameter tensors received finite gradients

逐层 shape trace（信息路径，不是构造函数）：
  module                         type               shape
  image_encoder.0.0              Conv2d             (32, 16, 64, 64)
  image_encoder.1.0              Conv2d             (32, 32, 32, 32)
  image_encoder.2.0              Conv2d             (32, 64, 16, 16)
  image_encoder.3.0              Conv2d             (32, 128, 8, 8)
  image_encoder.6                Linear             (32, 256)
  proprio_encoder.0              Linear             (32, 128)
  proprio_encoder.2              Linear             (32, 128)
  goal_encoder.0                 Linear             (32, 32)
  token_embedding                Embedding          (32, 8, 16)
  language_encoder.0             Linear             (32, 32)
  fusion.0                       Linear  

## 3.8.5.5 四个模态开关 = 3.8.6 的消融臂

模型带模态开关，另有一个 **`language_mode`**，用来在实现上区分 3.8.4.4 的 L0 与 L1：

| 臂 | 输入 | 语言表示 | 对应 |
|---|---|---|---|
| **state only** | `proprio + goal` | 无 | 3.7 的基线 |
| **A** | `image + proprio + goal` | 无 | 3.8.4.3 的含 `task_goal` 版本 |
| **B** | `image + proprio` | **L0 任务编号**（`Embedding(2,32)`） | 3.8.4.4 的**上界臂** |
| **C** | `image + proprio` | **L1 token embedding** | 3.8.4.4 的语言臂 |
| **C+goal** | `image + proprio + goal + language` | L1 | 本 notebook 训练的那一个 |

**B 与 C 是 3.8.4.4 "L0 封顶 L1" 的可执行版本。** 两个任务下 L0 与 L1 **信息等价**
（存在双射），所以预期 **B ≈ C**；若出现 **C > B**，说明实验坏了（例如 C 偷看了 `goal`）。

这里有一个我第一版写错、值得记下的点：**如果 language 通道用 token embedding，那么"任务编号臂"
与"语言臂"其实是同一段代码路径**（都是 `Embedding` → 32 维），参数量会完全相同——那样
"B 与 C 是两个臂"就是假的。所以模型加了 `language_mode`，让 L0 走 `Embedding(2,32)`、
L1 走 `Embedding(10,16)` + masked pool，两者才是**可比较的两个实现**。

**这也正是 language 必须用 token embedding 的理由**：`task_id` 模式下**没有词序**，
3.8.4.6 的 T2（打乱词序）**没有定义**。选 token 路径是**零信息代价**的（L0 ≡ L1），
但保住了三级阶梯。

注意**臂 A 与臂 C 的区别只有 `task_goal` 与 language 的互换**——这正是 3.8.4.5 说的
"把 `task_goal` 拿掉，指令就是唯一能区分任务的通道"。但 3.8.4.5 同时证明了**这个"唯一"对
`image` 不成立**（`t=0` 起 100%），所以 C 的实验是**预注册为否定**的。

In [11]:
# 3.8.5.5 模态开关
ARMS = {
    "state only  proprio+goal":  dict(use_image=False, use_language=False),
    "A  image+proprio+goal":     dict(use_language=False),
    "B  image+proprio+task-ID":  dict(use_goal=False, language_mode="task_id"),
    "C  image+proprio+language": dict(use_goal=False, language_mode="tokens"),
    "C+goal  (full)":            dict(language_mode="tokens"),
}

print(f"  {'arm':<27} {'fusion in':>9} {'params':>10}  forward")
models = {}
for name, kw in ARMS.items():
    m = mp.MultimodalPolicy(vocab_size=len(VOCAB), t_txt=T_TXT, **kw)
    out = m(batch)
    assert out.shape == (len(target), H, ACTION_DIM), (name, out.shape)
    models[name] = m
    print(f"  {name:<27} {sum(m.fusion_dims):>9} {mp.count_params(m):>10,}  {tuple(out.shape)}")

# B is L0 (one embedding per task), C is L1 (one embedding per word). Both carry the SAME
# information with two tasks (3.8.4.4), so they must fuse to the same dimension.
b_dims = sum(models["B  image+proprio+task-ID"].fusion_dims)
c_dims = sum(models["C  image+proprio+language"].fusion_dims)
assert b_dims == c_dims == 416, (b_dims, c_dims)
print()
print(f"  B and C both fuse to {b_dims}: same information, different implementation.")
print("  -> 3.8.4.4's 'L0 is a ceiling on L1' is now an executable comparison.")
print()
print("  task_id mode has NO word order, so 3.8.4.6's T2 is UNDEFINED for arm B. That is the")
print("  reason the language channel uses tokens: zero information cost (L0 = L1 with two")
print("  tasks), and the three-rung ladder stays defined.")

  arm                         fusion in     params  forward
  state only  proprio+goal          160    250,176  (32, 8, 8)
  A  image+proprio+goal             416    511,712  (32, 8, 8)
  B  image+proprio+task-ID          416    511,648  (32, 8, 8)
  C  image+proprio+language         416    512,288  (32, 8, 8)
  C+goal  (full)                    448    528,800  (32, 8, 8)

  B and C both fuse to 416: same information, different implementation.
  -> 3.8.4.4's 'L0 is a ceiling on L1' is now an executable comparison.

  task_id mode has NO word order, so 3.8.4.6's T2 is UNDEFINED for arm B. That is the
  reason the language channel uses tokens: zero information cost (L0 = L1 with two
  tasks), and the three-rung ladder stays defined.


## 3.8.5.6 训练

保持与 3.7 一致：**MSE loss、Adam、best-validation checkpoint、固定 seed**。

$$L=\frac{1}{H}\sum_{i=0}^{H-1}\lVert a_i-\hat a_i\rVert^2$$

**本 notebook 只训练一个模型**（全模态，臂 C+goal），目的是验证 pipeline 通。
三个臂的完整对比属于 3.8.6。

**预期它一定会过拟合**：**528,800 个参数对 509 个训练样本**。这不是缺陷，是这一节的定位——
3.8.5 回答"pipeline 对不对"，不回答"性能好不好"。所以下面的读数要按这个标准看。

In [12]:
# 3.8.5.6 训练（全模态臂；3.8.6 才做三臂对比）
EPOCHS = 150
train_loader = torch.utils.data.DataLoader(
    mp.DictDataset(data, train_mask), batch_size=32, shuffle=True)
val_loader = torch.utils.data.DataLoader(
    mp.DictDataset(data, val_mask), batch_size=64, shuffle=False)

torch.manual_seed(SEED)
model = mp.MultimodalPolicy(vocab_size=len(VOCAB), t_txt=T_TXT)
result = mp.train_model(model, train_loader, val_loader, epochs=EPOCHS, lr=1e-3, seed=SEED)

print()
print(f"best val MSE (normalised) = {result['best_val']:.6f} at epoch {result['best_epoch']}")

# 常数预测器基线：预测训练集的平均 chunk。低于它才说明学到了东西。
mean_chunk = data["action_chunk"][train_mask].reshape(-1, H * ACTION_DIM).mean(0)
mean_pred = np.broadcast_to(mean_chunk.reshape(H, ACTION_DIM),
                            (int(val_mask.sum()), H, ACTION_DIM))
baseline = float(((data["action_chunk"][val_mask] - mean_pred) ** 2).mean())
print(f"mean-action baseline       = {baseline:.6f}")
print(f"  -> model {'beats' if result['best_val'] < baseline else 'does NOT beat'} the baseline")

# 原始单位下的误差，便于和 3.7 的读数对照
a_mean, a_std = (t.numpy() for t in norm["action"])
raw_rmse = float(np.sqrt(result["best_val"]) * a_std.mean())
print(f"\nnormalised MSE {result['best_val']:.6f} <-> about {raw_rmse:.4f} in raw action units")
print(f"(H = {H} rows, so h=0 is the only fair single-step comparison -- lesson 3.7)")

  epoch    1  train 0.84768  val 0.46324   *
  epoch   25  train 0.00831  val 0.08386   *
  epoch   50  train 0.00475  val 0.09266   
  epoch   75  train 0.00406  val 0.10101   
  epoch  100  train 0.00337  val 0.10177   
  epoch  125  train 0.00276  val 0.11450   
  epoch  150  train 0.00268  val 0.12355   

best val MSE (normalised) = 0.083863 at epoch 25
mean-action baseline       = 0.597227
  -> model beats the baseline

normalised MSE 0.083863 <-> about 0.0651 in raw action units
(H = 8 rows, so h=0 is the only fair single-step comparison -- lesson 3.7)


### 读数：这里能得出什么、不能得出什么

**能得出的**：pipeline 通了。多模态输入 → 编码 → 融合 → `[B,8,8]`，反向能传播，
loss 在下降，而且**动作侧契约与 3.7 逐字节相同**。这是 3.8.6 做归因的前提。

**不能得出的**：

- **不能说"多模态有效"**——那需要三个臂在**同一划分、同一归一化、同一 seed** 下对比（3.8.6）。
- **不能说"模型使用了语言"**——这正是 3.8.4.6 的对象。而且我们已经**预注册了否定结果**：
  这个 pool 里 `image` 88.7%（`t=0` 起 100%）、`proprio` 从 `t=1` 起 98%、`task_goal` 10/10
  全都泄露任务，所以语言盲模型也能达到同样的离线水平。
- **不能说它学到了策略**——528,800 参数 / 509 样本，过拟合是预期内的。

### 预注册的预测（3.8.6 检验）

3.8.4.6 给出了三条判据。下面直接测，并且**把 T2 当作架构的阴性对照**。

**一个必须写清楚的细节。** 均值池化在**实数**上是置换不变的，但

$$\operatorname{pool}(z)=\frac{1}{n}\sum_i z_i$$

在浮点里**不是**：求和顺序随词序改变，舍入结果就不一样。所以 T2 的残差会是
**float32 舍入量级**而不是精确的 0。因此判据不能写成 `== 0`，要写成
**"相对 T3 可忽略"**——下面同时报残差、`eps` 和 `T2/T3` 比值。

$$\operatorname{sens}(o,\mathrm{T})=\big\|\pi(o,\ell_{\text{correct}})-\pi(o,\ell_{\mathrm{T}})\big\|_{h=0}$$

| 测试 | 预测 | 如果违反说明什么 |
|---|---|---|
| **T2** shuffled（词序） | **在浮点精度内为 0** | 均值池化不是置换不变的，或位置信息漏了进来 |
| **T3** contradictory（词袋） | ≠ 0 | 若为 0，语言**确实没被用**（本 pool 预期如此） |
| `drop`（全 `<pad>`） | 与 T3 同量级 | 模型对"没有指令"有自己的响应模式 |

In [13]:
# 3.8.5.7 三个反事实测试（判据来自 3.8.4.6）
print(f"  {'mode':<16} {'mean|d|':>10} {'gripper':>10}  前 3 个 arm 维")
print("-" * 62)
sens = {}
for mode in ("shuffled", "contradictory", "drop"):
    m, per_dim, signed = mp.instruction_sensitivity(model, data, val_mask, mode, horizon=0)
    sens[mode] = (m, per_dim)
    print(f"  {mode:<16} {m:>10.6f} {per_dim[7]:>10.6f}  {np.round(per_dim[:3], 6)}")

# T2 是架构的阴性对照。判据必须按浮点精度写，不能写 == 0：
# (x*keep).sum(1) 的求和顺序随词序改变，所以残差是舍入而不是对词序的真实依赖。
EPS32 = float(torch.finfo(torch.float32).eps)
t2, t3 = sens["shuffled"][0], sens["contradictory"][0]
print()
print(f"float32 eps = {EPS32:.3e}")
print(f"T2 residual = {t2:.3e}   ({t2 / EPS32:.2f} x eps)")
print(f"T3 movement = {t3:.3e}")
print(f"T2 / T3     = {t2 / t3:.2e}")

assert t2 < 1e-6, f"T2 residual {t2:.3e} is above float noise; word order genuinely leaks"
assert t2 / t3 < 1e-4, f"T2 is not negligible against T3 (ratio {t2 / t3:.2e})"
print()
print("T2 sits at float32 rounding level and is about six orders of magnitude smaller than T3,")
print("so word order does not reach the action: the permutation-invariance prediction of 3.8.4.6")
print("holds on a real model. T2 is therefore a negative control on the implementation, and only")
print("T3 can detect language use.")
print()
print(f"T3 moved the action by {t3:.3e}, so the model is not language-blind in the strict sense.")
print("On this pool that is expected to be a shortcut through task_goal and proprio rather than a")
print("reading of the instruction (3.8.4.5). The clean test is the t=0 proprio-only probe, which")
print("3.8.6 runs.")

  mode                mean|d|    gripper  前 3 个 arm 维
--------------------------------------------------------------
  shuffled           0.000000   0.000000  [0. 0. 0.]
  contradictory      0.114631   0.375897  [0.042748 0.075736 0.112344]
  drop               0.052475   0.155153  [0.025963 0.033996 0.049855]

float32 eps = 1.192e-07
T2 residual = 3.273e-08   (0.27 x eps)
T3 movement = 1.146e-01
T2 / T3     = 2.85e-07

T2 sits at float32 rounding level and is about six orders of magnitude smaller than T3,
so word order does not reach the action: the permutation-invariance prediction of 3.8.4.6
holds on a real model. T2 is therefore a negative control on the implementation, and only
T3 can detect language use.

T3 moved the action by 1.146e-01, so the model is not language-blind in the strict sense.
On this pool that is expected to be a shortcut through task_goal and proprio rather than a
reading of the instruction (3.8.4.5). The clean test is the t=0 proprio-only probe, which
3.8.6 ru

## 小结

1. **第一版是 Encoder + Concat Fusion + Action Head**，全模态 528,800 参数，
   `(image, proprio, goal, language) → action_chunk [8,8]`，动作侧与 3.7 逐字节相同。
2. **不用 cross-attention 是能力边界问题，不是练手。** 拼接式融合缺**两个**条件
   （token 网格 + attention 步），所以原理上做不了指称 grounding；而本数据唯一能测的正是
   动作模式 grounding——**能力边界与可测范围恰好对齐**。
3. **image 用从头训练的小 CNN**：pretrained 权重本机无缓存、需要联网（不可复现）；
   你的"不引入 pretrained distribution 变量"原则对它同样成立；11M 参数对 509 样本是保证过拟合，
   而且 ResNet18 你无法逐层解释。
4. **language 用 token embedding + masked mean pool，而不是 one-hot**：one-hot **没有词序**，
   会让 3.8.4.6 的 T2 失去定义、三级阶梯坍缩成两级；而 2 任务下 L0 ≡ L1，替换是**零信息代价**的。
5. **归一化必须跨任务 pooled**：夹爪的任务差异承载了 99.54% 的任务相关方差，按任务归一化会把它抹掉。
   这条现在有断言守着。
6. **`F.mse_loss` 的静默 broadcast 用断言挡**，不用注释挡——`[B,8,8]` 对 `[B,8]` 会算出一个
   `[B,B,8]` 的垃圾 loss 而不报错。
7. **T2 的置换不变性预测第一次落到真实模型上，并且通过了。** 但结论要写准：残差**不是**精确的 0，
   而是 **float32 舍入量级**（`4.8e-08`，`eps = 1.19e-07`），比 T3 小约 **10⁶ 倍**。
   原因是 `(x*keep).sum(1)` 的**求和顺序**随词序改变——**数学上置换不变，浮点里不是**。
   所以判据是"相对 T3 可忽略"，不是 `== 0`；写 `== 0` 的断言会失败（这一版我第一遍就写错了）。
8. **本 notebook 不回答"多模态是否有效"**，那是 3.8.6：同一划分、同一归一化、同一 seed，
   只换模态开关。

## 自检

1. 为什么第一版不用 cross-attention？请给出**两个**条件，并说明本架构各缺哪一个。
2. 为什么 image encoder 用从头训练的小 CNN，而不用 pretrained ResNet18？给出三条理由，
   并说明哪一条是"硬"的（即与环境有关而不是与偏好有关）。
3. one-hot task id 为什么会让 3.8.4.6 的三级阶梯坍缩？换成 token embedding 之后，
   模型的信息能力变强了吗？
4. 如果按任务分别归一化动作，夹爪那一路会发生什么？为什么这会毁掉语言实验？
5. `F.mse_loss` 在 `[B,8,8]` 与 `[B,8]` 之间会发生什么？为什么它很危险，断言怎么写？
6. T2（打乱词序）在本模型上必须是 0。这是**数据**的性质还是**架构**的性质？
   如果换成带 positional encoding 的 transformer 语言编码器，T2 会变成什么？
7. 本 notebook 训练出的模型，为什么**不能**用来宣称"多模态提升性能"？3.8.6 需要控制哪些变量？